In [1]:
import xarray as xr
import numpy as np
from osgeo import gdal

# same file and point as in extract_hk.py
l1_path = "/Users/tjdu/Desktop/ptree/H08_FLDK_2km_20210101/NC_H08_20210101_0000_R21_FLDK.06001_06001.nc"
CENTER_LON = 114.1576935
CENTER_LAT = 22.3507405
GRID_NUM_2KM = 50

ds2 = xr.open_dataset(l1_path)

# use the same variable and grid transform as your script
da = ds2["albedo_01"]  # any of your vars with (lat, lon)

lon_min = da["longitude"].values.min()
lon_max = da["longitude"].values.max()
lat_max = da["latitude"].values.max()
lat_min = da["latitude"].values.min()

gt = (lon_min, 0.02, 0, lat_max, 0, -0.02)
inv_gt = gdal.InvGeoTransform(gt)  # same as your code

def get_slices(inv_gt, point_lon, point_lat, grid_num):
    offsets = gdal.ApplyGeoTransform(inv_gt, point_lon, point_lat)
    xoff, yoff = map(int, offsets)
    if grid_num % 2 != 0:
        grid_offset = (grid_num - 1) / 2
        lu_x = int(xoff - grid_offset)
        lu_y = int(yoff - grid_offset)
        ld_y = int(yoff + grid_offset)
        ru_x = int(xoff + grid_offset)
        y_slice = slice(lu_y, ld_y + 1)
        x_slice = slice(lu_x, ru_x + 1)
    else:
        grid_offset = grid_num / 2
        lu_x = int(xoff - grid_offset)
        lu_y = int(yoff - grid_offset)
        ld_y = int(yoff + grid_offset)
        ru_x = int(xoff + grid_offset)
        y_slice = slice(lu_y, ld_y)
        x_slice = slice(lu_x, ru_x)
    return x_slice, y_slice

x_slice2, y_slice2 = get_slices(inv_gt, CENTER_LON, CENTER_LAT, GRID_NUM_2KM)

# now use those slices to get the 2 km coordinates EXACTLY as your script does
lat2 = ds2["latitude"].values
lon2 = ds2["longitude"].values

lat_sub2 = lat2[y_slice2]
lon_sub2 = lon2[x_slice2]

print("2 km get_grid-equivalent crop:")
print("  lat size:", lat_sub2.size)
print("  lon size:", lon_sub2.size)
print("  lat[0], lat[-1]:", float(lat_sub2[0]), float(lat_sub2[-1]))
print("  lon[0], lon[-1]:", float(lon_sub2[0]), float(lon_sub2[-1]))
print("  dlat:", float(lat_sub2[1] - lat_sub2[0]))
print("  dlon:", float(lon_sub2[1] - lon_sub2[0]))

2 km get_grid-equivalent crop:
  lat size: 50
  lon size: 50
  lat[0], lat[-1]: 22.860000610351562 21.880001068115234
  lon[0], lon[-1]: 113.63999938964844 114.6199951171875
  dlat: -0.020000457763671875
  dlon: 0.0200042724609375


/var/folders/gv/1g5bz3nn7v36rb5dn1q8wjj80000gn/T/ipykernel_59690/1470457717.py:11: FutureWarning: In a future version of xarray decode_timedelta will default to False rather than None. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  ds2 = xr.open_dataset(l1_path)


In [ ]:
from pathlib import Path
# --- your existing get_slices(inv_gt, point_lon, point_lat, grid_num) must already be defined ---

CENTER_LON = 114.1576935
CENTER_LAT = 22.3507405
GRID_NUM = 50

nc_path = Path("/Users/tjdu/Desktop/ptree/H08_FLDK_2km_20210101/NC_H08_20210101_0000_R21_FLDK.06001_06001.nc")
result_root = Path("/Users/tjdu/Desktop/ptree/crops_npy")
result_root.mkdir(parents=True, exist_ok=True)

vars1 = [f"albedo_{i:02d}" for i in range(1, 7)]
vars2 = [f"tbb_{i:02d}" for i in range(7, 17)]
vars3 = ["SAA", "SAZ", "sd_albedo_03", "SOA", "SOZ"]
vars4 = vars1 + vars2 + vars3

ds = xr.open_dataset(nc_path)

l = []
for var in vars4:
    if var not in ds:
        raise KeyError(f"Variable '{var}' not found in {nc_path.name}")

    da = ds[var]  # expects dims include ('latitude','longitude')

    lon_min = float(da["longitude"].values.min())
    lat_max = float(da["latitude"].values.max())

    # same geotransform approach you already use
    gt = (lon_min, 0.02, 0.0, lat_max, 0.0, -0.02)
    inv_gt = gdal.InvGeoTransform(gt)

    x_slice, y_slice = get_slices(inv_gt, CENTER_LON, CENTER_LAT, GRID_NUM)

    # crop using your slices
    cropped = da.isel(longitude=x_slice, latitude=y_slice)

    # optional sanity check
    if cropped.sizes["latitude"] != GRID_NUM or cropped.sizes["longitude"] != GRID_NUM:
        raise ValueError(
            f"{var}: crop shape {cropped.sizes['latitude']}x{cropped.sizes['longitude']} "
            f"!= {GRID_NUM}x{GRID_NUM}. Center may be near edge or grid spacing mismatch."
        )

    l.append(cropped)

final = xr.concat(l, pd.Index(vars4, name="band"))
final.name = "data"  # (band, latitude, longitude)

out_path = result_root / f"{nc_path.stem}.npy"
np.save(out_path, final.data.astype(np.float32))

ds.close()
print("Saved:", out_path, "shape:", final.data.shape)


Saved: /Users/tjdu/Desktop/ptree/crops_npy/NC_H08_20210101_0000_R21_FLDK.06001_06001.npy shape: (21, 50, 50)


/var/folders/gv/1g5bz3nn7v36rb5dn1q8wjj80000gn/T/ipykernel_59690/4021464592.py:22: FutureWarning: In a future version of xarray decode_timedelta will default to False rather than None. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  ds = xr.open_dataset(nc_path)


In [4]:
npy = np.load("/Users/tjdu/Desktop/ptree/crops_npy/NC_H08_20210101_0000_R21_FLDK.06001_06001.npy")

In [6]:
npy.shape

(21, 50, 50)

In [7]:
npy2 = np.load("/Users/tjdu/Desktop/ptree/2kmhk/202101/20210101_0000.npy")

In [8]:
npy2.shape

(21, 50, 50)

In [9]:
# compare npy and npy2
npy - npy2

array([[[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]],

       [[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]],

       [[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]],

       ...,

       [[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0.